# Phase 2: Building the Vector Search Engine (Retriever)

In Phase 1, I processed the NIST 800-207 guidelines and built my foundational database. Now, in Phase 2, I am building the "Search Engine" of my RAG architecture.

In this notebook, I am writing the custom logic (the Retriever) that will take a user's question, search through my local ChromaDB vector database, and pull out the most highly relevant text chunks for the AI to read.

### What to expect in this notebook:

* **Step 1: Installing Dependencies & Connecting Storage** – Setting up my Python environment, mounting Google Drive, and verifying that my Phase 1 database is ready to go.
* **Step 2: Building the Advanced Retriever Engine** – Writing the custom search logic. This includes **Pre-Retrieval** query rewriting (to make the search highly technical) and **Post-Retrieval** cross-encoder reranking (to aggressively filter and re-order the results).
* **Step 3: Testing the Advanced Search Pipeline** – Running a live, complex cybersecurity query to prove that the search engine successfully finds and ranks the best NIST chunks.
* **Step 4: Backing Up the Code to Google Drive** – Securely saving my custom search engine script (`retrieval.py`) so I can easily import it into the AI Generator in Phase 3.

### Step 1: Installing Dependencies & Connecting Storage

Before I can build the search engine, I need to set up my environment.

In this cell, I am:
1. **Installing Libraries:** Downloading the necessary tools for vector math and database management (like LangChain, Sentence-Transformers, and ChromaDB).
2. **Mounting Google Drive:** Connecting my Colab notebook to my personal Google Drive.
3. **Verifying the Database:** Running a quick check to ensure Python can successfully locate the `chroma_db_bge` vector database I generated back in Phase 1.

```
# This is formatted as code
```



In [1]:
# Step 1: Installing Dependencies & Connecting Storage

# Install required libraries for Phase 2
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers chromadb

from google.colab import drive
import os

# Mount Drive to access the Knowledge Base we built in Phase 1
drive.mount('/content/drive')

# Verify the database exists
db_path = '/content/drive/My Drive/ZTA_Project/chroma_db_bge'
if os.path.exists(db_path):
    print("Developer Log: Database successfully located on Google Drive.")
else:
    print("WARNING: Database not found. Check the file path.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

### Step 2: Building the Advanced Retriever Engine

Now for the heavy lifting. In this cell, I am writing all the logic for my custom search engine and saving it into a Python file (`retrieval.py`) so I can easily plug it into my pipeline later.

To make sure my system finds the absolute best NIST documents, I am implementing three advanced retrieval techniques here:
1. **Query Rewriting (Pre-Retrieval):** I use a small, fast AI (`Flan-T5`) to look at the user's casual question and automatically rewrite it into a highly technical cybersecurity search query.
2. **Vector Search (The Broad Net):** I use my BGE embedding model to quickly scan the entire ChromaDB database and pull the top 5 potentially relevant text chunks.
3. **Cross-Encoder Reranking (The Microscope):** Because standard vector search can sometimes make mistakes, I use a secondary, highly focused AI (`ms-marco`) to physically read those top 5 chunks, score them against the user's exact question, and re-order them so only the absolute best 3 are sent to the final generator.

In [2]:
#Step 2: Building the Advanced Retriever Engine
%%writefile retrieval.py
import torch
# from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class AdvancedRetriever:
    def __init__(self, db_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing Advanced Retriever on {self.device.upper()}...")

        # # 1. Load the Bi-Encoder (BGE Embeddings from Phase 1)
        # self.embeddings = HuggingFaceBgeEmbeddings(
        #     model_name="BAAI/bge-large-en-v1.5",
        #     model_kwargs={'device': self.device},
        #     encode_kwargs={'normalize_embeddings': True}
        # )
        self.embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': self.device},
    encode_kwargs={'normalize_embeddings': True}
)

        # Load ChromaDB
        self.db = Chroma(persist_directory=db_path, embedding_function=self.embeddings)

        # 2. Load the Cross-Encoder for Reranking (Post-Retrieval Filter)
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=self.device)

        # 3. Load Query Rewriter (Pre-Retrieval)
        # FIX: Bypassing the 'pipeline' entirely and loading the model explicitly
        print("Loading Query Rewriter directly...")
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
        self.rewriter_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(self.device)

    def rewrite_query(self, original_query):
        """TECHNIQUE 1: Query Rewriting (HyDE concept)"""
        prompt = f"Rewrite this user question into a technical cybersecurity search query focusing on Zero Trust Architecture: {original_query}"

        # Generate the expanded query using raw tokenization
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.rewriter_model.generate(**inputs, max_length=50)
        expanded_query = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return expanded_query

    def retrieve_and_rerank(self, user_query, top_k_initial=5, top_k_final=3):
        """TECHNIQUE 2 & 3: Bi-Encoder Retrieval and Cross-Encoder Reranking"""

        # Step 1: Rewrite the query
        search_query = self.rewrite_query(user_query)

        # Step 2: Vector Search (The Broad Net)
        initial_docs = self.db.similarity_search(search_query, k=top_k_initial)

        # Step 3: Reranking (The Microscope)
        pairs = [[user_query, doc.page_content] for doc in initial_docs]
        scores = self.reranker.predict(pairs)

        scored_docs = list(zip(initial_docs, scores))
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        final_docs = [doc for doc, score in scored_docs[:top_k_final]]

        return {
            "original_query": user_query,
            "rewritten_query": search_query,
            "documents": final_docs
        }

Writing retrieval.py


### Step 3: Testing the Advanced Search Pipeline

With the custom logic built, it is time to run a live test. In this cell, I initialize my new `AdvancedRetriever` and simulate a complex, real-world cybersecurity question to see how the system performs.

To demonstrate the sophistication of this architecture, notice how the output showcases two critical AI operations:
* **Pre-Retrieval (Query Rewriting):** Before ever touching the database, the system takes my casual question and expands it into a highly technical search string to get better vector matches.
* **Post-Retrieval (Cross-Encoder Reranking):** After the initial search, the system doesn't just trust the database. It actively scores and re-orders the text chunks, ensuring only the absolute highest-quality context is passed to the final output.

Let's execute the pipeline and look at the results!

In [3]:
# Step 3: Testing the Advanced Search Pipeline
# Import the module we just created
from retrieval import AdvancedRetriever

# Initialize the engine using the path to your Google Drive DB
db_path = '/content/drive/My Drive/ZTA_Project/chroma_db_bge'
retriever = AdvancedRetriever(db_path)

# Simulate a High-Level Architect Query
test_query = "How do I prevent lateral movement if a remote worker's laptop is compromised?"

print("\n--- Executing Advanced Retrieval Pipeline ---\n")
results = retriever.retrieve_and_rerank(test_query)

print(f"Original Query:  {results['original_query']}")
print(f"Rewritten Query: {results['rewritten_query']}\n")

print("--- Top 3 Reranked Context Chunks ---")
for i, doc in enumerate(results['documents']):
    source = doc.metadata.get('source_document', 'Unknown Source')
    print(f"\n[Result {i+1} | Source: {source}]")
    print(doc.page_content[:400] + "...\n")

Initializing Advanced Retriever on CUDA...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


--- Executing Advanced Retrieval Pipeline ---

Original Query:  How do I prevent lateral movement if a remote worker's laptop is compromised?
Rewritten Query: What is the best way to prevent lateral movement if a remote worker's laptop is compromised?

--- Top 3 Reranked Context Chunks ---

[Result 1 | Source: NIST_800-207]
collection of resources once on the internal network. As a result, unauthorized lateral movement 
within the environment has been one of the biggest challenges for federal agencies.  
The Trusted Internet Connections (TIC) and agency perimeter firewalls provide strong internet 
gateways. This helps block attackers from the internet, but the TICs and perimeter firewalls are 
less useful for detecti...


[Result 2 | Source: NIST_800-53]
Discussion:  Monitoring and control of remote access methods allows organizations to 
detect attacks and help ensure compliance with remote access policies by auditing the 
connection activities of remote users on a variety of system 

### Step 4: Backing Up the Code to Google Drive

Google Colab environments are temporary, meaning any local files I create here (like the `retrieval.py` script I just wrote) will be completely erased as soon as my session disconnects.

To make sure I don't lose my hard work, this quick cell takes that script and securely copies it over to my persistent Google Drive folder. This is a crucial step because I will need to import this exact file later when I build the AI Generator in Phase 3 and run the automated evaluations in Phase 4!

In [4]:
# Step 4: Backing Up the Code to Google Drive
import shutil
import os

# Define the source and destination
source_file = 'retrieval.py'
drive_destination = '/content/drive/My Drive/ZTA_Project/retrieval.py'

# Copy the file to your Google Drive project folder
shutil.copy(source_file, drive_destination)

print(f"Developer Log: Code successfully backed up to {drive_destination}")

Developer Log: Code successfully backed up to /content/drive/My Drive/ZTA_Project/retrieval.py


### Phase 2 Summary: The Advanced Search Engine

In this phase, I upgraded the system from a basic vector database into an **Advanced RAG** pipeline. I built a modular retrieval engine (`retrieval.py`) designed to filter out noise and pull only the most accurate "ground truth" documents for the LLM.

**Key Techniques Implemented:**

* **Query Rewriting (Pre-Retrieval):** I integrated a lightweight sequence-to-sequence model (`google/flan-t5-small`) to automatically translate the user's raw question into a targeted search query. This helps the system search based on technical intent rather than just matching random words.
* **Bi-Encoder Search (Vector Retrieval):** I used the BGE embedding model to query the ChromaDB I built in Phase 1. This acts as a broad net, quickly scanning all 4,200+ chunks to grab the Top-5 most semantically similar paragraphs.
* **Cross-Encoder Reranking (Post-Retrieval Filter):** I implemented a dedicated reranking model (`ms-marco-MiniLM-L-6-v2`). This acts as a strict filter. It reads the 5 retrieved chunks alongside the user's original query, scores them based on true logical relevance, and discards the bottom 2.

**Result:** The pipeline is fully operational. It successfully takes a user query, translates it, searches the database, and outputs the absolute best Top-3 context chunks, complete with source metadata. This clean data package is now ready to be sent to the generation model in Phase 3.